# Pipeline 3 (HOG+SVM + Pinhole) & Final Combined Evaluation
**Owner:** Member 3 (Ali) | **Branch:** `pipeline-3` (and `main` after merge)

This notebook has two stages:
1. **Stage A (On your branch):** Trains the classical baseline (`HOG + Linear SVM` Pedestrian detector + `Pinhole Camera Model`) and saves it to Google Drive.
2. **Stage B (After merging all branches into `main`):** Collects all team weights (`yolov10n`, `yolov8n`, `hog_svm`) into `checkpoints/` and runs the full test evaluation (`04_evaluate_all.py --multi-seed-summary`) and speed benchmark (`05_benchmark_speed.py`) on a single GPU.

In [ ]:
!nvidia-smi
from google.colab import drive
import os

drive.mount('/content/drive')
SHARED_DRIVE = '/content/drive/MyDrive/CS423_Shared_Project'
os.makedirs(f'{SHARED_DRIVE}/checkpoints', exist_ok=True)
os.makedirs(f'{SHARED_DRIVE}/results', exist_ok=True)
print('Google Drive backup folder ready:', SHARED_DRIVE)

In [ ]:
REPO_URL = "https://github.com/nehirvardar/cs423-assistive-navigation.git"
BRANCH = "pipeline-3"  # Switch to "main" for final combined evaluation

!rm -rf /content/project
!git clone -b {BRANCH} {REPO_URL} /content/project
%cd /content/project
!pip install -q -r requirements.txt
!apt-get update -qq && apt-get install -qq -y aria2

In [ ]:
# Full KITTI download and data preparation
!python scripts/download_kitti.py
!python scripts/01_prepare_data.py

In [ ]:
# STAGE A: Train classical HOG + Linear SVM Pedestrian detector and back up to Drive
!python scripts/02_train_hog.py
!cp -v checkpoints/hog_svm_pedestrian.joblib {SHARED_DRIVE}/checkpoints/

In [ ]:
# STAGE B (Final Post-Merge Evaluation): Copy all team checkpoints from Drive
!mkdir -p checkpoints
!cp -v {SHARED_DRIVE}/checkpoints/* checkpoints/

# Run unified 3-pipeline evaluation on test split + 3-seed mean+/-std + BBox Ablation
!python scripts/04_evaluate_all.py --split test --multi-seed-summary --device cuda

# Benchmark end-to-end Latency (ms) and FPS on a single GPU
!python scripts/05_benchmark_speed.py --warmup 20 --runs 100 --device cuda

# Save final tables and plots to Google Drive
!cp -r results {SHARED_DRIVE}/